In [12]:
import pandas as pd
import requests

In [27]:
df = pd.read_csv('C:/Users/Marcelo/Desktop/Medicamentos/extracao-dados-medicamentos/airflow/dags/csvs/DADOS_ABERTOS_MEDICAMENTOS.csv', encoding="ISO-8859-1", delimiter=";", decimal=",")

In [28]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 31416 entries, 0 to 31415
Data columns (total 11 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   TIPO_PRODUTO                31416 non-null  object 
 1   NOME_PRODUTO                31416 non-null  object 
 2   DATA_FINALIZACAO_PROCESSO   30881 non-null  object 
 3   CATEGORIA_REGULATORIA       29539 non-null  object 
 4   NUMERO_REGISTRO_PRODUTO     31366 non-null  float64
 5   DATA_VENCIMENTO_REGISTRO    30746 non-null  object 
 6   NUMERO_PROCESSO             31416 non-null  int64  
 7   CLASSE_TERAPEUTICA          31314 non-null  object 
 8   EMPRESA_DETENTORA_REGISTRO  31416 non-null  object 
 9   SITUACAO_REGISTRO           31416 non-null  object 
 10  PRINCIPIO_ATIVO             11758 non-null  object 
dtypes: float64(1), int64(1), object(9)
memory usage: 2.6+ MB


In [30]:
df['EMPRESA_DETENTORA_REGISTRO'].apply(_extract_laboratory_code_from_enterprise_value)

0        60874187000184
1        63826069000199
2        61286647000116
3        92762277000170
4        00925697000101
              ...      
31411    88630413000109
31412    09240065000340
31413    60448040000122
31414    14864868000144
31415    60448040000122
Name: EMPRESA_DETENTORA_REGISTRO, Length: 31416, dtype: object

In [23]:
df2 = df.copy()
df2['codigo_laboratorio'] = df['EMPRESA_DETENTORA_REGISTRO'].apply(_extract_laboratory_code_from_enterprise_value)
df2['nome_laboratorio'] =  df['EMPRESA_DETENTORA_REGISTRO'].apply(_extract_laboratory_name_from_enterprise_value)
df2 = df2[[k for k in COLUMN_MAPPER.keys()]]
df2 = df2.rename(columns=COLUMN_MAPPER) 

In [24]:
COLUMN_MAPPER = {
    'TIPO_PRODUTO': 'tipo_produto',
    'NOME_PRODUTO': 'nome',
    'CATEGORIA_REGULATORIA': 'categoria_regulatoria',
    'CLASSE_TERAPEUTICA': 'classe_terapeutica',
    'NUMERO_REGISTRO_PRODUTO' : 'numero_registro',
    'PRINCIPIO_ATIVO': 'principio_ativo',
    'codigo_laboratorio': 'codigo_laboratorio',
    'nome_laboratorio': 'nome_laboratorio',
}

In [25]:
df2

,tipo_produto,nome,categoria_regulatoria,classe_terapeutica,numero_registro,principio_ativo,codigo_laboratorio,nome_laboratorio
0,MEDICAMENTO,(VITAMINAS A ) + ASSSOCIACÕES,SIMILAR,VITAMINAS E SUPLEMENTOS MINERAIS,104540166.0,NaN,60874187000184,DAIICHI SANKYO BRASIL FARMACÊUTICA LTDA
1,MEDICAMENTO,AC SALICILICO + AC BENZOICO + IODO,SIMILAR,ANTIMICOTICOS PARA USO TOPICO,119350001.0,NaN,63826069000199,LABORATORIO FLORA DA AMAZONIA LTDA
2,MEDICAMENTO,ALENDRONATO SODICO,SIMILAR,SUPRESSORES DA REABSORCAO OSSEA,100470305.0,NaN,61286647000116,SANDOZ DO BRASIL INDÚSTRIA FARMACÊUTICA LTDA
3,MEDICAMENTO,ARNICA MONTANA,FITOTERÁPICO,FITOTERAPICO SIMPLES,104730021.0,NaN,92762277000170,VIDORA FARMACÊUTICA LTDA
4,MEDICAMENTO,ARNICA MONTANA L.,FITOTERÁPICO,FITOTERAPICO SIMPLES,131750004.0,NaN,00925697000101,LIMED LABORATORIO INDUSTRIAL DE MEDICAMENTOS L...
...,...,...,...,...,...,...,...,...
31411,MEDICAMENTO,Florbetaben (18F),RADIOFÁRMACO,NaN,NaN,NaN,88630413000109,UNIÃO BRASILEIRA DE EDUCAÇÃO E ASSISTÊNCIA
31412,MEDICAMENTO,Gallium-68,RADIOFÁRMACO,NaN,NaN,OCTREOTIDA NOTA ALUMÍNIO (18 F),09240065000340,R2 SOLUÇÕES EM RADIOFARMÁCIA
31413,MEDICAMENTO,PIB (11C),RADIOFÁRMACO,AGENTES DIAGNOSTICOS RADIOATIVOS,NaN,PIBENZOTIAZOL (11 C),60448040000122,HOSPITAL DAS CLINICAS DA FACULDADE DE MEDICINA...
31414,MEDICAMENTO,PSMA - 1007 (18F),RADIOFÁRMACO,AGENTES DIAGNOSTICOS RADIOATIVOS,NaN,AMESPRO FLUORNICOTINAMIDA (18 F),14864868000144,IBF


In [20]:
def _extract_laboratory_code_from_enterprise_value(enterprise_value: str):
    if pd.isnull(enterprise_value):
        return
    if '-' not in enterprise_value:
        return None
    return enterprise_value.split('-')[0].strip()

In [21]:
def _extract_laboratory_name_from_enterprise_value(enterprise_value: str):
    if pd.isnull(enterprise_value):
        return
    if '-' not in enterprise_value:
        return enterprise_value
    return enterprise_value.split('-')[1].strip()

In [30]:
file = requests.get('https://dados.anvisa.gov.br/dados/DADOS_ABERTOS_MEDICAMENTOS.csv')

SSLError: HTTPSConnectionPool(host='dados.anvisa.gov.br', port=443): Max retries exceeded with url: /dados/DADOS_ABERTOS_MEDICAMENTOS.csv (Caused by SSLError(SSLCertVerificationError(1, '[SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: unable to get local issuer certificate (_ssl.c:1002)')))